<a href="https://colab.research.google.com/github/okpast/python-data-analysis-projects/blob/main/ab_test_significance_analysis/ab_test_significance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Підключення до BigQuery та формування датасету

### 1.1. Встановлення та імпорт бібліотек

In [ ]:
RUN_INITIAL_SETUP = True

if RUN_INITIAL_SETUP:
  # Встановлення/оновлення бібліотеки для роботи з BigQuery
  !pip install --upgrade -q google-cloud-bigquery

# Імпорт необхідних бібліотек
from google.colab import auth
from google.cloud import bigquery
import numpy as np
import pandas as pd

### 1.2. Авторизація та підключення до BigQuery

In [ ]:
if RUN_INITIAL_SETUP:
  # Авторизація облікового запису Google для доступу до BigQuery
  auth.authenticate_user()

  # Створення клієнта BigQuery для проєкту Google Cloud
  project_id = 'data-analytics-mate'
  client = bigquery.Client(project=project_id)

### 1.3. Виконання SQL-запиту та формування датасету

In [ ]:
if RUN_INITIAL_SETUP:
  # Запис SQL-запиту у змінну
  query = """
  WITH session_info AS (
    SELECT  s.ga_session_id,
            s.date,
            sp.country,
            sp.continent,
            sp.device,
            sp.channel,
            ab.test,
            ab.test_group
    FROM    `data-analytics-mate.DA.ab_test` ab
    JOIN    `data-analytics-mate.DA.session` s
    ON      ab.ga_session_id = s.ga_session_id
    JOIN    `data-analytics-mate.DA.session_params` sp
    ON      s.ga_session_id = sp.ga_session_id
  ),

  session_agg AS (
    SELECT  date, country, continent, device, channel, test, test_group,
            COUNT(DISTINCT ga_session_id) AS session_cnt
    FROM    session_info
    GROUP BY ALL
  ),

  account_agg AS (
    SELECT  si.date, si.country, si.continent, si.device, si.channel, si.test, si.test_group,
            COUNT(DISTINCT acs.account_id) AS new_account_cnt
    FROM    `data-analytics-mate.DA.account_session` acs
    JOIN    session_info si
    ON      acs.ga_session_id = si.ga_session_id
    GROUP BY ALL
  ),

  session_order_agg AS (
    SELECT  si.date, si.country, si.continent, si.device, si.channel, si.test, si.test_group,
            COUNT(DISTINCT o.ga_session_id) AS session_with_order_cnt
    FROM    `data-analytics-mate.DA.order` o
    JOIN    session_info si
    ON      o.ga_session_id = si.ga_session_id
    GROUP BY ALL
  ),

  event_agg AS (
  SELECT  si.date, si.country, si.continent, si.device, si.channel, si.test, si.test_group,
          ep.event_name,
          COUNT(ep.ga_session_id) AS event_cnt
  FROM    `data-analytics-mate.DA.event_params` ep
  JOIN    session_info si
  ON      ep.ga_session_id = si.ga_session_id
  GROUP BY ALL
  )

  SELECT  date, country, continent, device, channel, test, test_group,
          'session' AS event_name,
          session_cnt AS value
  FROM    session_agg

  UNION ALL

  SELECT  date, country, continent, device, channel, test, test_group,
          'new_account' AS event_name,
          new_account_cnt AS value
  FROM    account_agg

  UNION ALL

  SELECT  date, country, continent, device, channel, test, test_group,
          'session_with_orders' AS event_name,
          session_with_order_cnt AS value
  FROM    session_order_agg

  UNION ALL

  SELECT  date, country, continent, device, channel, test, test_group,
          event_name,
          event_cnt AS value
  FROM    event_agg;
  """

  # Виконання запиту та завантаження результатів у DataFrame
  # df = client.query(query).to_dataframe()

  # Виконання SQL-запиту та завантаження результатів у DataFrame.
  # BigQuery Storage API вимкнено через відсутність дозволу
  # 'bigquery.readsessions.create' у поточному проєкті.
  # Результати завантажуються через стандартний BigQuery API
  df = client.query(query).to_dataframe(create_bqstorage_client=False)

## 2. Первинний аналіз та перевірка якості даних

### 2.1. Загальна інформація про датасет

In [ ]:
# Перевірка розмірності датасету
print("Розмірність датасету:", df.shape)

# Назви стовпців, типи даних і кількість непорожніх значень
print("\nІнформація про датасет:")
df.info()

# Вивід перших 5 рядків
print("\nПерші 5 рядків датасету:")
display(df.head())

Розмірність датасету: (800996, 9)

Інформація про датасет:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   continent   800996 non-null  object
 3   device      800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB

Перші 5 рядків датасету:


,date,country,continent,device,channel,test,test_group,event_name,value
0,2020-11-02,(not set),Asia,desktop,Organic Search,2,1,new_account,1
1,2020-11-17,(not set),Asia,mobile,Organic Search,2,2,new_account,1
2,2020-11-19,(not set),Asia,desktop,Undefined,2,1,new_account,1
3,2020-11-28,(not set),Asia,mobile,Organic Search,2,2,new_account,1
4,2020-11-01,(not set),Americas,desktop,Direct,1,2,new_account,1


### 2.2. Перевірка пропущених значень і дублікатів

In [ ]:
# Кількість пропущених значень у кожному стовпці
display(df.isna().sum().to_frame(name='missing_values'))

# Кількість повністю дубльованих рядків
print("\nКількість дублікатів:", df.duplicated().sum())

,missing_values
date,0
country,0
continent,0
device,0
channel,0
test,0
test_group,0
event_name,0
value,0



Кількість дублікатів: 0


### 2.3. Перевірка унікальності та кількості значень

In [ ]:
# Кількість унікальних значень у кожному стовпці
display(df.nunique(dropna=False).to_frame(name='unique_values'))

,unique_values
date,88
country,108
continent,6
device,3
channel,5
test,4
test_group,2
event_name,19
value,936


### 2.4. Перевірка та коригування типів даних

In [ ]:
# Перевірка типів даних
print("Типи даних до коригування:")
print(df.dtypes)

# Перетворення date на стандартний тип datetime pandas
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Перетворення текстових стовпців з повторюваними значеннями на тип category
categorical_columns = ["country", "continent", "device", "channel"]
df[categorical_columns] = df[categorical_columns].astype("category")

# Перевірка типів даних після коригування
print("\nТипи даних після коригування:")
print(df.dtypes)

# Перевірка кількості некоректних дат
print("\nКількість некоректних дат:", df['date'].isna().sum())

# Перевірка відсутності пропущених значень у стовпці value
assert df['value'].notna().all(), 'NA in value column'

Типи даних до коригування:
date          dbdate
country       object
continent     object
device        object
channel       object
test           Int64
test_group     Int64
event_name    object
value          Int64
dtype: object

Типи даних після коригування:
date          datetime64[ns]
country             category
continent           category
device              category
channel             category
test                   Int64
test_group             Int64
event_name            object
value                  Int64
dtype: object

Кількість некоректних дат: 0


### 2.5. Перевірка діапазону дат і значень

In [ ]:
print("Мінімальна дата:", df['date'].min())
print("Максимальна дата:", df['date'].max())

# Статистичний опис стовпця value
display(df['value'].describe())

# Кількість нульових і від'ємних значень
print("\nНульові значення:", (df['value'] == 0).sum())
print("Від'ємні значення:", (df['value'] < 0).sum())

Мінімальна дата: 2020-11-01 00:00:00
Максимальна дата: 2021-01-27 00:00:00


,value
count,800996.0
mean,9.475863
std,37.493267
min,1.0
25%,1.0
50%,2.0
75%,5.0
max,1575.0



Нульові значення: 0
Від'ємні значення: 0


### 2.6. Висновки

- **Структура датасету:** набір даних містить 800 996 рядків і 9 стовпців.
- **Пропущені значення:** відсутні в усіх стовпцях.
- **Дублікати:** повністю дубльованих рядків не виявлено.
- **Унікальні значення:** датасет охоплює 88 дат, 108 країн, 6 континентальних категорій, 3 типи пристроїв, 5 каналів, 4 ідентифікатори тестів, 2 тестові групи та 19 типів подій.
- **Типи даних:** дату приведено до типу `datetime` pandas, а текстові поля з повторюваними значеннями оптимізовано за допомогою типу `category`.
- **Діапазон дат:** дані охоплюють період з 1 листопада 2020 року до 27 січня 2021 року.
- **Значення метрик:** стовпець `value` містить лише додатні значення в діапазоні від 1 до 1575. Нульових і від'ємних значень не виявлено. Медіана становить 2, середнє значення - приблизно 9,48, стандартне відхилення - приблизно 37,49. Середнє значення помітно вище за медіану, що вказує на правосторонню асиметрію розподілу: відносно невелика кількість великих значень підвищує середнє.

## 3. Розрахунок статистичної значущості A/B-тестів

### 3.1. Визначення метрик і функції Z-тесту

Метрики визначено за умовою завдання за формулою `подія / session`.

- Для `add_payment_info`, `add_shipping_info` і `begin_checkout` чисельником є загальна кількість відповідних подій. Одна сесія може містити кілька подій одного типу, тому значення показника може перевищувати 100%.
- Для `new_account` чисельником є кількість унікальних облікових записів, створених у реєстраційних сесіях.

Двопропорційний Z-тест передбачає незалежні бінарні спостереження. Для показників, що враховують загальну кількість повторюваних подій, ця умова формально не виконується, оскільки події в межах однієї сесії можуть бути залежними. Тому результати такого тесту для цих показників слід інтерпретувати з обережністю: статистична значущість може бути завищеною.

Якщо значення показника перевищує 100% хоча б в одній групі, Z-тест не застосовується: Z-статистика та p-value не обчислюються, а результат не використовується для висновку про статистичну значущість.

Рядки, для яких `p-value` не було обчислено (`NaN`), виключено з підсумкової таблиці результатів і збережено в окремому файлі `ab_test_significance_excluded_rows.csv` для подальшого аналізу.

In [ ]:
from scipy import stats

ALPHA = 0.05
MIN_N = 500

# Метрики та відповідні події-чисельники й події-знаменники
metrics = {
    'add_payment_info':     ('add_payment_info',    'session'),
    'add_shipping_info':    ('add_shipping_info',   'session'),
    'begin_checkout':       ('begin_checkout',      'session'),
    'new_account':          ('new_account',         'session'),
}

def z_test(x_c, n_c, x_v, n_v):

  # x_c, n_c — події та розмір вибірки контрольної групи (control)
  # x_v, n_v — події та розмір вибірки тестової групи (variant)

  # Перевірка розмірів вибірок
  if n_c <= 0 or n_v <= 0:
    return np.nan, np.nan, np.nan, np.nan

  # Перевірка кількості подій
  if not (0 <= x_c <= n_c and 0 <= x_v <= n_v):
    return x_c / n_c, x_v / n_v, np.nan, np.nan

  # Конверсії контрольної та тестової груп
  p_c = x_c / n_c
  p_v = x_v / n_v

  # Об'єднана пропорція за нульової гіпотези
  p_pool = (x_c + x_v) / (n_c + n_v)

  # Стандартна похибка різниці пропорцій
  se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_c + 1 / n_v))

  # Z-статистика та двостороннє p-значення
  if se == 0:
    return p_c, p_v, np.nan, np.nan

  z = (p_v - p_c) / se
  p_value = 2 * stats.norm.sf(abs(z))

  return p_c, p_v, z, p_value

### 3.2. Розрахунок результатів A/B-тестів за сегментами

In [ ]:
def calc_results(df, metrics, alpha=ALPHA, min_n=MIN_N):

  # Створюємо копію вихідного DataFrame, щоб не змінювати df
  data = df.copy()

  # Відбираємо події, необхідні для розрахунку метрик
  needed = set().union(*metrics.values())
  data = data[data['event_name'].isin(needed)].copy()


  # Перетворюємо категоріальні стовпці на рядковий тип
  for col in ['continent', 'device', 'channel', 'event_name']:
    data[col] = data[col].astype(str)

  # Перетворюємо числові стовпці на цілочисельний тип
  for col in ['test', 'test_group', 'value']:
    data[col] = pd.to_numeric(data[col], errors='raise').astype('int64')

  # Формуємо списки тестів і значень вимірів для сегментного аналізу
  tests = sorted(data['test'].unique())
  continents = ['All'] + sorted(x for x in data['continent'].unique() if x != 'All')
  devices = ['All'] + sorted(x for x in data['device'].unique() if x != 'All')
  channels = ['All'] + sorted(x for x in data['channel'].unique() if x != 'All')

  # Створюємо список для збереження результатів розрахунків
  rows = []

  # Перебираємо всі тести та комбінації сегментів
  for test in tests:
    data_test = data[data['test'] == test]

    for cont in continents:
      # Відбираємо дані для поточного континенту або для всіх континентів
      data_cont = (data_test if cont == 'All' else data_test[data_test['continent'] == cont])

      for dev in devices:
        # Відбираємо дані для поточного пристрою або для всіх пристроїв
        data_dev = (data_cont if dev == 'All' else data_cont[data_cont['device'] == dev])

        for ch in channels:
          # Відбираємо дані для поточного каналу або для всіх каналів
          data_ch = (data_dev if ch == 'All' else data_dev[data_dev['channel'] == ch])

          # Групуємо за тестовою групою та назвою події, підсумовуючи value
          agg = data_ch.groupby(['test_group', 'event_name'])['value'].sum()

          # Послідовний розрахунок кожної метрики
          for metric, (num, den) in metrics.items():
            # Отримуємо чисельники та знаменники
            # для контрольної і тестової груп
            try:
              x_c = int(agg.loc[(1, num)])
              n_c = int(agg.loc[(1, den)])
              x_v = int(agg.loc[(2, num)])
              n_v = int(agg.loc[(2, den)])

            # Пропускаємо комбінації, для яких немає необхідних даних
            except KeyError:
              continue

            # Пропускаємо комбінації без сесій в одній із груп
            if n_c == 0 or n_v == 0:
              continue

            # Розраховуємо конверсії, z-статистику та p-value
            p_c, p_v, z, p = z_test(x_c, n_c, x_v, n_v)

            # Відносна зміна показника тестової групи, %
            change_pct = (p_v - p_c) / p_c * 100 if p_c != 0 else np.nan

            # Перевіряємо статистичну значущість
            significant = bool(p < alpha) if not np.isnan(p) else None

            # Позначаємо недостатній розмір вибірки
            low_sample = min(n_c, n_v) < min_n

            # Формуємо рядок із результатами поточної метрики
            row = {
                'test_number': test,
                'continent': cont,
                'device': dev,
                'channel': ch,
                'metric': metric,
                'numerator_event': num,
                'denominator_event': den,

                'numerator_count_control': x_c,
                'denominator_count_control': n_c,
                'conversion_rate_control': p_c,

                'numerator_count_variant': x_v,
                'denominator_count_variant': n_v,
                'conversion_rate_variant': p_v,

                'metric_change_pct': change_pct,

                # Результати перевірки статистичної значущості
                'z_stat': z,
                'p_value': p,
                'significant': significant,
                'low_sample': low_sample,
            }

            # Зберігаємо результат
            rows.append(row)

  # Повертаємо DataFrame з результатами розрахунків
  return pd.DataFrame(rows)

### 3.3. Фільтрація та експорт результатів для Tableau

In [ ]:
EXPORT_TO_TABLEAU = True

if EXPORT_TO_TABLEAU:
  import os
  from google.colab import drive

  # Підключаємо Google Drive, якщо він ще не підключений
  if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

  # Створюємо папку для збереження результатів
  out = (
      '/content/drive/MyDrive/Mate_Academy/'
      'Colab_Notebooks/ab_test_significance_analysis/'
  )
  os.makedirs(out, exist_ok=True)


  # Розраховуємо результати для чотирьох метрик
  results = calc_results(df, metrics, min_n=MIN_N)

  # Визначаємо причини виключення рядків
  conditions = [
      # results['low_sample'].fillna(False),
      ~results['p_value'].between(0, 1, inclusive='both')
  ]

  reasons = [
      # 'insufficient_sample_size',
      'invalid_p_value'
  ]

  results['exclusion_reason'] = np.select(
      conditions,
      reasons,
      default=''
  )


  # Відбираємо рядки, що підлягають виключенню
  excluded = results[results['exclusion_reason'] != ''].copy()

  # Виводимо кількість виключених рядків
  print(f"Виключено рядків: {len(excluded)} із {len(results)}")

  # Переглядаємо виключені рядки
  display(excluded)

  # Зберігаємо виключені рядки для подальшого аналізу
  excluded.to_csv(out + 'ab_test_significance_excluded_rows.csv', index=False)


  # Залишаємо лише валідні результати для Tableau
  results = results[results['exclusion_reason'] == ''].copy()

  # Видаляємо службовий стовпець із причинами виключення
  results = results.drop(columns='exclusion_reason')

  # Скидаємо індекс після фільтрації
  results = results.reset_index(drop=True)

  # Перетворюємо значення significant на булевий тип
  results['significant'] = results['significant'].astype(bool)

  # Зберігаємо результати для візуалізації в Tableau
  results.to_csv(out + 'ab_test_significance_results.csv', index=False)

  # Виводимо розмірність набору даних для Tableau
  print('\nРозмір таблиці результатів:', results.shape)

  # Перевіряємо вміст папки з результатами
  print('Файли в папці:', os.listdir(out))
else:
  print("Експорт пропущено. Встанови EXPORT_TO_TABLEAU = True, щоб запустити.")

Виключено рядків: 3 із 2076


,test_number,continent,device,channel,metric,numerator_event,denominator_event,numerator_count_control,denominator_count_control,conversion_rate_control,numerator_count_variant,denominator_count_variant,conversion_rate_variant,metric_change_pct,z_stat,p_value,significant,low_sample,exclusion_reason
113,1,(not set),desktop,Paid Search,begin_checkout,begin_checkout,session,1,8,0.1250,15,13,1.153846,823.076923,NaN,NaN,None,True,invalid_p_value
1130,3,(not set),All,Undefined,begin_checkout,begin_checkout,session,17,16,1.0625,10,8,1.250000,17.647059,NaN,NaN,None,True,invalid_p_value
1139,3,(not set),desktop,Undefined,begin_checkout,begin_checkout,session,13,10,1.3000,10,5,2.000000,53.846154,NaN,NaN,None,True,invalid_p_value



Розмір таблиці результатів: (2073, 18)
Файли в папці: ['ab_test_significance_excluded_rows.csv', 'ab_test_significance_results.csv']


## 4. Висновки

### 4.1. Загальні результати A/B-тестів

In [ ]:
# Відбір результатів на загальному рівні без сегментації
# за континентом, пристроєм і каналом залучення
verification_all = results[
    (results['continent'] == 'All') &
    (results['device'] == 'All') &
    (results['channel'] == 'All')
].copy()

# Сортування за номером тесту, статистичною значущістю
# та величиною зміни метрики у відсотках
verification_all = verification_all.sort_values(
    ['test_number', 'significant', 'metric_change_pct'],
    ascending=[True, False, False]
)

# Відбір стовпців для перевірки результатів
verification_all = verification_all[[
    'test_number',
    'metric',
    'numerator_count_control',
    'denominator_count_control',
    'conversion_rate_control',
    'numerator_count_variant',
    'denominator_count_variant',
    'conversion_rate_variant',
    'metric_change_pct',
    'p_value',
    'significant'
]].copy()

# Перетворення конверсій у відсотки
verification_all['conversion_rate_control'] *= 100
verification_all['conversion_rate_variant'] *= 100

# Округлення числових показників
verification_all = verification_all.round({
    'conversion_rate_control': 3,
    'conversion_rate_variant': 3,
    'metric_change_pct': 2,
    'p_value': 4
})

# Перейменування стовпців для відображення
verification_all = verification_all.rename(columns={
    'test_number': 'Тест',
    'metric': 'Метрика',
    'numerator_count_control': 'Події_control',
    'denominator_count_control': 'Сесії_control',
    'conversion_rate_control': 'Конверсія_control_%',
    'numerator_count_variant': 'Події_variant',
    'denominator_count_variant': 'Сесії_variant',
    'conversion_rate_variant': 'Конверсія_variant_%',
    'metric_change_pct': 'Відносна_зміна_%',
    'p_value': 'p-value',
    'significant': 'Статистично_значуща'
})

display(verification_all)

,Тест,Метрика,Події_control,Сесії_control,Конверсія_control_%,Події_variant,Сесії_variant,Конверсія_variant_%,Відносна_зміна_%,p-value,Статистично_значуща
0,1,add_payment_info,1988,45362,4.383,2229,45193,4.932,12.54,0.0001,True
2,1,begin_checkout,3784,45362,8.342,4021,45193,8.897,6.66,0.0029,True
1,1,add_shipping_info,3034,45362,6.688,3221,45193,7.127,6.56,0.0092,True
3,1,new_account,3823,45362,8.428,3681,45193,8.145,-3.35,0.1229,False
506,2,add_payment_info,2344,50637,4.629,2409,50244,4.795,3.58,0.2146,False
508,2,begin_checkout,4262,50637,8.417,4313,50244,8.584,1.99,0.3406,False
507,2,add_shipping_info,3480,50637,6.872,3510,50244,6.986,1.65,0.4780,False
509,2,new_account,4165,50637,8.225,4184,50244,8.327,1.24,0.5560,False
1017,3,begin_checkout,9532,70047,13.608,9264,70439,13.152,-3.35,0.0120,True
1015,3,add_payment_info,3623,70047,5.172,3697,70439,5.249,1.47,0.5201,False


- **Тест 1.** Номінально значуще зростання показників `add_payment_info` (+12.54%, `p-value = 0.0001`), `begin_checkout` (+6.66%, `p-value = 0.0029`) та `add_shipping_info` (+6.56%, `p-value = 0.0092`). Для `new_account` (-3.35%, `p-value = 0.1229`) статистично значущої зміни не виявлено.

- **Тест 2.** Для всіх чотирьох метрик (`add_payment_info`, `begin_checkout`, `add_shipping_info`, `new_account`) статистично значущої зміни не виявлено (`p-value > 0.05`).

- **Тест 3.** Номінально значуще зниження `begin_checkout` (-3.35%, `p-value = 0.0120`). Для `add_payment_info` (+1.47%, `p-value = 0.5201`), `new_account` (-1.13%, `p-value = 0.5199`) та `add_shipping_info` (-2.62%, `p-value = 0.1574`) значущої зміни не виявлено.

- **Тест 4.** Номінально значуще зниження `begin_checkout` (-2.35%, `p-value = 0.0459`) та `new_account` (-3.36%, `p-value = 0.0175`). Для `add_shipping_info` (-3.41%, `p-value = 0.0741`) і `add_payment_info` (-3.54%, `p-value = 0.1162`) значущої зміни не виявлено. Усі чотири показники в цьому тесті змінилися в бік зниження.

**Загальний висновок.** Номінально статистично значущими є шість із 16 перевірок: три в тесті 1, одна в тесті 3 та дві в тесті 4. Після застосування поправки Бонферроні (поріг 0.05 / 16 = 0.003125) значущими залишаються лише `add_payment_info` (`p-value = 0.0001`) і `begin_checkout` (`p-value = 0.0029`) у тесті 1. Решта чотири номінально значущі результати не проходять цей поріг.

Показники `add_payment_info`, `add_shipping_info` і `begin_checkout` розраховуються на основі загальної кількості подій. Оскільки одна подія може повторюватися в межах однієї сесії, такі спостереження не є незалежними. Двовибірковий Z-тест пропорцій передбачає бінарний результат для кожної одиниці спостереження, тому його застосування до цих метрик може призводити до некоректних p-value та завищення статистичної значущості. Зростання показників у тесті 1 може відображати збільшення кількості повторних подій у межах сесій, а не збільшення частки сесій, у яких відбулася відповідна подія.

Єдиний номінально значущий результат за метрикою `new_account`, для якої повторення події не спостерігалися, — у тесті 4 (−3.36%). Однак він не проходить поправку Бонферроні (p-value = 0.0175 > 0.003125).

### 4.2. Результати сегментного аналізу

In [ ]:
# Відбір сегментних результатів:
# хоча б один вимір (континент, пристрій або канал) відрізняється від All;
# виключаємо сегменти з недостатнім розміром вибірки
verification_segment = results[
    (
        (results['continent'] != 'All') |
         (results['device'] != 'All') |
        (results['channel'] != 'All')
    ) &
    (~results['low_sample'])
].copy()

print(f"Сегментних результатів після фільтра low_sample: {len(verification_segment)}")

# Сортування за тестом, значущістю та величиною зміни
verification_segment = verification_segment.sort_values(
    ['test_number', 'significant', 'metric_change_pct'],
    ascending=[True, False, False]
)

# Формування таблиці для перегляду
verification_segment = verification_segment[
    [
        'test_number', 'continent', 'device', 'channel', 'metric',
        'numerator_count_control', 'denominator_count_control',
        'numerator_count_variant', 'denominator_count_variant',
        'metric_change_pct', 'p_value', 'significant'
    ]
].copy()

# Округлення числових показників
verification_segment['metric_change_pct'] = verification_segment['metric_change_pct'].round(2)
verification_segment['p_value'] = verification_segment['p_value'].round(4)

# Перейменування стовпців
verification_segment = verification_segment.rename(columns={
    'test_number': 'Тест',
    'continent': 'Континент',
    'device': 'Пристрій',
    'channel': 'Канал',
    'metric': 'Метрика',
    'numerator_count_control': 'Події_control',
    'denominator_count_control': 'Сесії_control',
    'numerator_count_variant': 'Події_variant',
    'denominator_count_variant': 'Сесії_variant',
    'metric_change_pct': 'Зміна_%',
    'p_value': 'p-value',
    'significant': 'Статистично_значуща'
})

Сегментних результатів після фільтра low_sample: 1136


In [ ]:
# Формування висновків за кожним тестом

TOP_N = 3

for test in sorted(verification_segment['Тест'].unique()):
  test_data = verification_segment[
      verification_segment['Тест'] == test
  ]
  significant_data = test_data[
      test_data['Статистично_значуща'].eq(True)
  ]

  positive = significant_data[
      significant_data['Зміна_%'] > 0
  ].nlargest(TOP_N, 'Зміна_%')

  negative = significant_data[
      significant_data['Зміна_%'] < 0
  ].nsmallest(TOP_N, 'Зміна_%')


  # Очікувана кількість хибнопозитивних результатів,
  # якщо всі нульові гіпотези правильні
  expected = len(test_data) * ALPHA

  print(f"\n{'=' * 80}")
  print(f'ТЕСТ {test}')
  print(f"{'=' * 80}")
  print(f"Усього сегментних результатів: {len(test_data)}")

  print(
      f"Номінально значущих результатів: "
      f"{int(test_data['Статистично_значуща'].sum())} "
      f"(очікується випадково ≈ {expected:.1f}, якщо всі нульові гіпотези правильні)"
  )
  print(f"Серед них зростань: {(significant_data['Зміна_%'] > 0).sum()}")
  print(f"Серед них знижень: {(significant_data['Зміна_%'] < 0).sum()}")


  print('\nНайбільші номінально значущі зростання:')
  if positive.empty:
    print('Номінально значущих зростань не виявлено.')
  else:
    for _, row in positive.iterrows():
      print(f"- {row['Метрика']}: +{row['Зміна_%']:.2f}%; "
            f"{row['Континент']} + {row['Пристрій']} + {row['Канал']}; "
            f"сесії: {row['Сесії_control']} / {row['Сесії_variant']}; "
            f"p-value = {row['p-value']:.4f}")

  print("\nНайбільші номінально значущі зниження:")
  if negative.empty:
    print('Номінально значущих знижень не виявлено.')
  else:
    for _, row in negative.iterrows():
      print(f"- {row['Метрика']}: {row['Зміна_%']:.2f}%; "
            f"{row['Континент']} + {row['Пристрій']} + {row['Канал']}; "
            f"сесії: {row['Сесії_control']} / {row['Сесії_variant']}; "
            f"p-value = {row['p-value']:.4f}")


ТЕСТ 1
Усього сегментних результатів: 264
Номінально значущих результатів: 101 (очікується випадково ≈ 13.2, якщо всі нульові гіпотези правильні)
Серед них зростань: 71
Серед них знижень: 30

Найбільші номінально значущі зростання:
- add_payment_info: +150.72%; Europe + mobile + Direct; сесії: 799 / 810; p-value = 0.0000
- add_payment_info: +139.66%; Americas + mobile + Undefined; сесії: 778 / 742; p-value = 0.0000
- begin_checkout: +124.59%; Asia + All + Undefined; сесії: 733 / 838; p-value = 0.0000

Найбільші номінально значущі зниження:
- add_payment_info: -43.04%; Asia + desktop + Paid Search; сесії: 1566 / 1664; p-value = 0.0019
- add_payment_info: -37.89%; Asia + desktop + Organic Search; сесії: 2132 / 2223; p-value = 0.0016
- add_payment_info: -36.06%; All + tablet + All; сесії: 999 / 1009; p-value = 0.0459

ТЕСТ 2
Усього сегментних результатів: 268
Номінально значущих результатів: 68 (очікується випадково ≈ 13.4, якщо всі нульові гіпотези правильні)
Серед них зростань: 41
Сере

- **Тест 1.** Із 264 сегментних результатів номінально значущими є 101 (очікується випадково ≈ 13): 71 зростання і 30 знижень. Найбільше зростання: `add_payment_info` +150,72% у сегменті `Europe` + `mobile` + `Direct` (`p-value < 0.0001`). Найбільше зниження: `add_payment_info` −43,04% у сегменті `Asia` + `desktop` + `Paid Search` (`p-value = 0.0019`).

- **Тест 2.** Із 268 результатів номінально значущими є 68 (очікується ≈ 13): 41 зростання і 27 знижень. Найбільше зростання: `add_shipping_info` +109,44% у сегменті `Oceania` + `All` + `All` (`p-value = 0.0018`). Найбільше зниження: `add_payment_info` −44,45% у сегменті `Europe` + `mobile` + `Organic Search` (`p-value = 0.0037`).

- **Тест 3.** Із 288 результатів номінально значущими є 65 (очікується ≈ 14): 27 зростань і 38 знижень. Три найбільші зростання належать одному сегменту `Americas` + `mobile` + `Social Search`: `add_payment_info` +89,23%, `begin_checkout` +73,36%, `add_shipping_info` +65,47%. Найбільше зниження: `add_payment_info` −55,12% у сегменті `Asia` + `desktop` + `Social Search` (`p-value < 0.0001`).

- **Тест 4.** Із 316 результатів номінально значущими є 118 (очікується ≈ 16): 36 зростань і 82 зниження. Три найбільші зростання належать одному сегменту `Asia` + `mobile` + `Direct`: `add_payment_info` +79,50%, `add_shipping_info` +72,36%, `begin_checkout` +68,63%. Найбільше зниження: `add_payment_info` −64,09% у сегменті `Asia` + `tablet` + `All` (`p-value = 0.0005`).

**Загальний висновок.** Із 1 136 сегментних результатів 352 (31,0%) є номінально значущими, тоді як за умови правильності всіх нульових гіпотез і коректності тесту випадково очікується ≈ 56,8 хибнопозитивних результатів. Таке перевищення саме по собі не свідчить про наявність реальних ефектів у сегментах.

Показники `add_payment_info`, `add_shipping_info` і `begin_checkout` розраховуються на основі загальної кількості подій. Оскільки події можуть повторюватися в межах однієї сесії, їх не можна вважати незалежними бінарними спостереженнями. Тому застосований двовибірковий Z-тест пропорцій може призводити до некоректних p-value та завищення статистичної значущості. Крім того, сегментний аналіз не скориговано на множинні порівняння. Отже, результати мають дослідницький характер, а висновки про конкретні сегменти потребують додаткової перевірки методами, що враховують повторюваність і залежність подій у межах сесій.


## Results

- 🔗 [A/B Test Significance Analysis — Tableau Dashboard](https://public.tableau.com/app/profile/volodymyr.tsapko/viz/ABTestSignificanceAnalysis_17925911138990/Dashboard)
- 📄 [A/B Test Significance Analysis — Calculation Results (CSV)](https://drive.google.com/file/d/1piZnmhib2LesaBBLf7PGQ66xO9-3QDOt/view?usp=sharing)